In [ ]:
# CELL 1 - SETUP: copy the code from your private Kaggle code dataset, install it, show the GPU
import glob, importlib, os, shutil, subprocess, sys

# The code dataset is the unzipped dist/cloud_code/genai_code.zip (rebuild + upload a NEW VERSION after code changes).
# Kaggle may mount it under a nested folder, so look for tools/t4_pipeline.py anywhere under /kaggle/input.
found = glob.glob("/kaggle/input/**/tools/t4_pipeline.py", recursive=True)
assert found, "code dataset not found (or an old version without tools/t4_pipeline.py): Add Input / update the dataset"
CODE_DIR = os.path.dirname(os.path.dirname(found[0]))
print("code found in", CODE_DIR)
import re
print("code version:", re.search(r'PIPELINE_VERSION = "([^"]+)"', open(found[0]).read()).group(1))   # the docs say which value is expected

# always refresh the code; dirs_exist_ok keeps the artifacts/ folder of an earlier run, so a resume finds its state
shutil.copytree(CODE_DIR, "/kaggle/working/repo", dirs_exist_ok=True)
os.chdir("/kaggle/working/repo")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "-r", "requirements_kaggle.txt"])
importlib.invalidate_caches()
subprocess.call(["nvidia-smi"])


In [ ]:
# CELL 2 - PARAMETERS: the only cell you normally edit
DEVICE_PROFILE = "kaggle"     # picks configs/devices/kaggle.yaml (data under /kaggle/input, outputs in /kaggle/working)
TAG = "t4"                    # names the log/state folder artifacts/logs/<TAG>/ (keep it when you re-run to RESUME)
DRY_RUN = True                # True = tiny end-to-end check (a few minutes); then set False for the real run
WANDB_SECRET = "WANDB_API_KEY"   # name of the Kaggle secret that holds your Weights & Biases API key


In [ ]:
# CELL 3 - RUN: the supervised Task 4 pipeline
#   data check -> benchmark -> final cGAN training (resumes) -> split checkpoint -> ONNX export + parity -> val evaluation
# No logic lives in this notebook; tools/t4_pipeline.py restarts crashed stages and resumes where it stopped.
# BEFORE a real run: set train.epochs in configs/task4_final.yaml (docs/TASK4_PLAN.md C17), rebuild the code zip.
import os, subprocess, sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(WANDB_SECRET)
except Exception as err:
    print("No W&B API key found in the Kaggle secrets (", err, ") -> the runner will log offline.")

cmd = [sys.executable, "tools/t4_pipeline.py", "--device-profile", DEVICE_PROFILE, "--tag", TAG, "--num-workers", "4"] + (["--dry-run"] if DRY_RUN else [])
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in proc.stdout:
    print(line, end="")
print("runner exit code:", proc.wait())


In [ ]:
# CELL 4 - OUTPUTS: bundle everything you need into ONE file, then download it (Output tab after Save Version, or the file panel)
import shutil
from pathlib import Path

root, stage = Path("/kaggle/working/repo"), Path("/kaggle/working/t4_results")
shutil.rmtree(stage, ignore_errors=True)
stage.mkdir(parents=True)
items = ["wandb", "models/onnx", "artifacts/promote_t4", "report/tables/onnx_parity.csv", "docs/BENCHMARKS.md", f"artifacts/logs/{TAG}",
         "artifacts/dryrun_t4/logs", "artifacts/dryrun_t4/eval",          # exist only after a dry run
         "/kaggle/working/runs", "/kaggle/working/eval"]                  # final run (ckpt_best/last, metrics, samples) + evaluation
for item in items:
    src = Path(item) if item.startswith("/") else root / item
    if src.exists():
        dst = stage / (src.relative_to("/kaggle/working") if item.startswith("/") else src.relative_to(root))
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)
shutil.make_archive("/kaggle/working/t4_results", "zip", stage)
print(f"{Path('/kaggle/working/t4_results.zip').stat().st_size / 2**20:.1f} MB: /kaggle/working/t4_results.zip")
log_dir = root / ("artifacts/dryrun_t4/logs" if DRY_RUN else f"artifacts/logs/{TAG}")
print((log_dir / "t4_status.log").read_text()[-3000:])
